# Erdős discrepancy problem

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order
@njit
def calculate_discrepancy(sequence: tuple[int, ...]) -> tuple[int, int]:
  """Calculates the discrepancy of a given sign sequence."""
  max_discrepancy = 0
  violating_seq_count = 0
  length = len(sequence)
  for d in range(1, length + 1):
    # if the sequence length is divisible by d
    if length % d == 0:
      subsequence_sum = 0
      for i in range(d - 1, length, d):
        subsequence_sum += sequence[i]
      max_discrepancy = max(max_discrepancy, abs(subsequence_sum))
      if abs(subsequence_sum) > 2:
        violating_seq_count += 1
  return max_discrepancy, violating_seq_count


def calculate_discrepancy_score(sequence: List[int]) -> float:
  """Calculates a score based on the length of the initial discrepancy 2 subsequence."""
  if not sequence:
    return 0.0
  sequence = sequence[1:]
  for i in range(len(sequence)):
    val = sequence[i]
    if val != 1 and val != -1:
      return -1_000_000.0
  tuple_sequence = tuple(sequence)
  for length in range(1, len(tuple_sequence) + 1):
    initial_subsequence = tuple_sequence[:length]
    discrepancy, violating_seq_count = calculate_discrepancy(
        initial_subsequence
    )
    if discrepancy > 2:
      # We went too far, the discrepancy increased at this length
      return float(length - 1) + 1.0 / (violating_seq_count + 1)
  # If the entire sequence has discrepancy <= 2
  return float(len(tuple_sequence))


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation in code (same as 2D)."""
  formatted_feedback = {}
  np.set_printoptions(threshold=np.inf)
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)
      array_content = cleaned_repr_str[6:-1]
      if np.iscomplexobj(value):
        formatted_feedback[key] = (
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'
    elif isinstance(value, list):
      formatted_feedback[key] = repr(value)
    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(params) -> tuple[dict[str, float], dict[str, str]]:
  """Evaluates sign sequences for the Erdős discrepancy problem (C=2)."""
  result = {}
  feedback = {}
  del params  # We are not using any input parameters for this search
  best_sequence = search_for_long_low_discrepancy_sequence()
  score = calculate_discrepancy_score(best_sequence)
  result['score'] = score
  feedback['best_score_found'] = score
  feedback['low_discrepancy_sequence'] = best_sequence
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""FunSearch experiment codebase for the Erdős discrepancy problem (C=2)."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import random
import re
from typing import Any, Callable, Mapping, List, Tuple
import scipy.linalg as la
import collections
import copy
import math
import numba

njit = numba.njit


def search_for_long_low_discrepancy_sequence():
  """Searches for long sign sequences with discrepancy at most 2."""
  max_sequence_length = 1124
  variable_name = 'low_discrepancy_sequence_iqhd'
  if variable_name in globals():
    current_sequence = list(globals()[variable_name])
  else:
    current_sequence = [
        random.choice([1, -1]) for _ in range(random.randint(1, 20))
    ]

  best_sequence = current_sequence[:]
  best_score = calculate_discrepancy_score(best_sequence)
  print(f'Initial score: {best_score}, sequence: {best_sequence[:10]}...')

  start_time = time.time()
  eval_count = 0
  while time.time() - start_time < np.random.randint(100, 1000):
    # Mutate the current sequence (add, remove, or flip a sign)
    if len(current_sequence) < max_sequence_length and random.random() < 0.3:
      index_to_insert = random.randint(0, len(current_sequence))
      current_sequence.insert(index_to_insert, random.choice([1, -1]))
    elif current_sequence and random.random() < 0.2:
      index_to_remove = random.randint(0, len(current_sequence) - 1)
      current_sequence.pop(index_to_remove)
    elif current_sequence:
      index_to_flip = random.randint(0, len(current_sequence) - 1)
      current_sequence[index_to_flip] *= -1

    score = calculate_discrepancy_score(current_sequence)
    eval_count += 1

    if score > best_score:
      best_score = score
      best_sequence = current_sequence[:]
      print(
          f'Best score: {best_score}, length: {len(best_sequence)}, sequence:'
          f' {best_sequence}...'
      )

    if random.random() < 0.1 and best_sequence:
      current_sequence = best_sequence[:]

  print(f'Final best score: {best_score}, final length: {len(best_sequence)}')
  print(f'Evaluations: {eval_count}')
  return best_sequence


def make_sequence_multiplicative(sequence: list[int]) -> list[int]:
  """Makes a sequence multiplicative."""
  if not sequence:
    return []
  # the new sequence will have the same length as the original sequence
  multiplicative_sequence = sequence[:]
  for a in range(1, int(np.sqrt(len(sequence)))):
    for b in range(a, len(sequence)):
      if a * b >= len(sequence):
        break
      multiplicative_sequence[a * b] = sequence[a] * sequence[b]
  return multiplicative_sequence

**Prompt used**

The Erdős Discrepancy Problem (C=2)

Act as an expert in number theory and combinatorial mathematics, to solve a
specific instance of the Erdős discrepancy problem. The problem asks whether
for any infinite $\pm 1$-sequence $(x_1, x_2, \ldots)$ and any integer $C$,
there exist integers $k$ and $d$ such that $|\sum_(i=1)^k x_(i \cdot d)| > C$.

We are focusing on the case where $C=2$. Your task is to find a $\pm 1$-sequence
that has a discrepancy of at most 2 for as long as possible. The discrepancy of
a sequence $(x_1, x_2, \ldots, x_n)$ is defined as the maximum absolute value of
the sum of any subsequence of the form $(x_d, x_(2d), \ldots, x_(kd))$ where $d$
is a positive integer and $kd \leq n$.

Your task is to produce a search function that generates $\pm 1$-sequences
(sequences where each element is either 1 or -1) and tries to find long
sequences that maintain a discrepancy of at most 2.

You will be evaluated based on the length of the longest initial subsequence
that has a discrepancy of 2. The scoring function will reward longer initial
sequences with discrepancy 2. If a sequence exceeds a discrepancy of 2 at a
certain length, the score will be approximately that length, with a small
penalty based on how much the discrepancy exceeds 2 at the next element.

The scoring function you will use is as follows:

def calculate_discrepancy_score(sequence: List[int]) -> float:
    """Calculates a score based on the length of the initial discrepancy 2
    subsequence."""
    if not sequence:
        return 0.0
    tuple_sequence = tuple(sequence)
    for length in range(1, len(tuple_sequence) + 1):
        initial_subsequence = tuple_sequence[:length]
        discrepancy = calculate_discrepancy(initial_subsequence)
        if discrepancy > 2:
            # We went too far, the discrepancy increased at this length
            return float(length - 1) + 1.0 / (
                calculate_discrepancy(tuple_sequence[:length]) + 1e-9)
    # If the entire sequence has discrepancy <= 2
    return float(len(tuple_sequence))

You may code up any search method you want, and you are allowed to call the
calculate_discrepancy_score() and calculate_discrepancy() functions as many
times as you want. You want the score it gives you to be as high as possible!

Your task is to write a search function that searches for the best
$\pm 1$-sequence.
Your function will have 1000 seconds to run, and after that, it has to have
returned the best construction it found. The maximum length of the sequence you
should aim for is around 1124.

Extremely important hint by Terence Tao: try functions which are multiplicative, or approximately multiplicative. Since our arrays are indexed from zero, this means we want our construction to satisfy best_sequence[a]best_sequence[b]=best_sequence[ab] for all a, b.

A couple more hints Terence Tao told us, that could help you:

Near-Multiplicativity: The sequences that almost provide counterexamples (like modified Dirichlet characters $\tilde(\chi)$) are often completely multiplicative ($f(mn)=f(m)f(n)$) or close to it. Consider exploring sequences that have some multiplicative structure. For instance, try defining $f(p)$ for primes $p$ and extending it multiplicatively, perhaps with specific choices for $f(p)$ when $p$ divides a small number $q$ (e.g., $f(3)=+1$ in the $\tilde(\chi)_3$ example).
Relation to Characters: Functions related to Dirichlet characters $\chi \pmod q$ (functions that are periodic modulo $q$ and multiplicative) are known to have low discrepancy (if zeros are handled). Explore sequences that mimic the structure of $\chi(n)$ for small $q$ (like $q=3, 4, \dots$), potentially by setting $f(n) = \chi(n)$ when $\chi(n) \neq 0$ and $f(n) = \pm 1$ when $\chi(n) = 0$.
Avoid Strong "Pretension": While structures related to characters are good starting points, the analysis suggests true counterexamples (if they existed) or very long low-discrepancy sequences might need to avoid looking too much like a simple character $\chi(n)$ or a modulated character $\chi(n)n^(it)$. This hints that the optimal sequence might possess a more complex structure than simple periodicity or multiplicativity.
Consider structures related to Walsh functions or other Fourier-analytic concepts on the Boolean hypercube. For example, sequences that try to "cancel out" correlations measured by low-degree Walsh functions might have low discrepancy.
Random Multiplicative Model: Instead of deterministic structures like characters, consider sequences built like random multiplicative functions. Here's a simple model: assign $x_p = \pm 1$ independently and randomly for each prime p, and then define $x_n = \prod x_p^(a_i)$ for $n = \prod p_i^(a_i)$. While a purely random sequence is unlikely to work perfectly, FunSearch could explore mutations or variations starting from this random multiplicative framework, potentially finding non-obvious structures with good cancellation properties.
The previously found best sequence is in the "low_discrepancy_sequence_iqhd" global variable, which you can access in your code and use it, it's probably a good idea to start your search from this sequence.

## What AlphaEvolve found

Without human guidance, AlphaEvolve found a sign pattern of length 200 with discrepancy 2 before progress slowed down. When given the hint to try multiplicative or approximately multiplicative functions, it performed much better, finding constructions of length 380. Nevertheless, these attempts were still far from the known optimal value of $C(2) = 1160$.